In [ ]:
%%html
<style>
    h1 {color:darkblue}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. **Part 1 — OpenAI Python APIs**
   * 01-01. Text Generation Via the Responses API
   * 01-02. Speech Recognition, Speech Synthesis, and Closed Captions
   * 01-03. **Images: Generation and Style Transfer**
   * 01-04. Content Moderation
   * 01-05. Generating Code with a Codex Model and the Responses API
2. Part 2 — OpenAI Agents SDK
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Image Generation and Editing
* Image workflows can use the **Images API** endpoints and the **Responses API image generation tool**
    * Create new images from text prompts
    * Edit existing images 
    * Mask-based edits for targeted replacement/inpainting
    * Generate variations of existing image
    * Can stream partial images to show progress (not in this presentation)
* `gpt-image-2` model ― OpenAI's latest image-generation model
    * Can customize `size`, `quality`, output format, ... 
* Examples
    1. Generate original images from text prompts
    2. Text-to-image style transfer via Images API `edit` endpoint and a prompt 
    3. Image-to-image style transfer via Responses API and image generation tool 
---


## `import` Modules and Setup `OpenAI` Client Object


In [ ]:
import IPython
from openai import OpenAI 
from pathlib import Path
client = OpenAI() # create the OpenAI client object

<hr/>


## 1. Generating Original Images
* Create new images from text prompts

### `create_image.py` Function
* [Open create_image.py](./source/create_image.py)
* Calls `images.generate`
    * Returns a JSON array of objects ― one by default
    * Received in Python code as a `list` of `dict`s
    * Request 1–10 images via `n` parameter
    * Image bytes are returned in Base64-encoded format
    * Stored in the corresponding `dict`'s `b64_json` attribute

In [ ]:
from source.create_image import create_image

#### Havanese Anime

In [ ]:
anime_path = Path('resources') / 'outputs' / 'havanese_anime.png'

In [ ]:
create_image(client, anime_path,
    """Havanese dog as a Japanese anime character
    in neon colors against a black background""")

In [ ]:
IPython.display.Image(anime_path, width=250)

#### Havanese Van Gogh

In [ ]:
vangogh_path = Path('resources') / 'outputs' / 'havanese_vangogh.png'

In [ ]:
create_image(client, vangogh_path,
    'Havanese dog in the style of Vincent van Gogh')

In [ ]:
IPython.display.Image(vangogh_path, width=250)

#### Havanese Davinci

In [ ]:
davinci_path = Path('resources') / 'outputs' / 'havanese_davinci.png'

In [ ]:
create_image(client, davinci_path,
    'Havanese dog in the style of Leonardo DaVinci')

In [ ]:
IPython.display.Image(davinci_path, width=250)

#### 
---

## `gpt-image-2` Size/Quality Reference 
* `size` — `'auto'` or `'WIDTHxHEIGHT'`
    * e.g., `'1024x1024'`, `'1536x1024'`, `'1024x1536'`, `'2048x2048'`, `'2048x1152'`, `'3840x2160'`, `'2160x3840'`
    * Edges must be **multiples of 16**
    * Aspect ratio between **3:1** and **1:3**
    * Maximum edge length **3840 px**
    * Sizes above `2560x1440` or `1440x2560` are **experimental**
    * Size options vary by model
    * `'auto'` lets model choose orientation 
        * Use explicit size for predictable layout 
* `quality` — `'low'`, `'medium'`, `'high'`, or `'auto'`
    * Use `low` for drafting to save money
* Cost/Latency
    * Bigger sizes and higher quality are more tokens
    * https://developers.openai.com/api/docs/guides/image-generation#cost-and-latency
---

## 2. Style Transfer via Text Prompt Using the Images API 
* Restyle an existing image with a detailed text prompt
* This example sends the source image plus a style-transfer prompt to the Images API `edit` endpoint

---

### Image to Restyle

In [ ]:
IPython.display.Image('./resources/sunset.jpg', width=500)

---

### Style-Transfer Prompt
* I used ChatGPT to create this descriptive prompt from a prior generated version of the **Havanese dog in the style of Vincent van Gogh**
* Note specific guidelines for
    * Medium/technique
    * Palette
    * Composition
    * Lighting & finish
    * Do/Don't
* Use direct verbs in your prompts
    * `generate`, `draw`, `edit`, `restyle`, or `transform`.
* For edits, state what should change and what should stay recognizable

In [ ]:
style_transfer_prompt = """
    Restyle the input photo into a vibrant swirling impasto 
    painting inspired by post-impressionist brushwork.  
    
    Medium/technique: thick acrylic paint with bold 
    palette-knife swipes and loaded brush strokes; swirling 
    arcs, rhythmic curves, comma-shaped dabs, and layered 
    ridges that give a tactile sheen (impasto).  
    
    Palette: luminous cobalt and ultramarine blues as the 
    dominant field; strong accents of golden yellow and amber; 
    secondary touches of teal and turquoise; minimal orange and 
    white highlights for contrast.  
    
    Composition: shallow depth, decorative and poster-flat; 
    energetic all-over brushwork that simplifies the subject 
    into flowing, abstracted shapes; swirls and curved strokes
    define contours and fur without precise detail.  
    
    Lighting & finish: very saturated, high contrast, minimal 
    shading; painterly, non-photorealistic.  
    
    Do/Don't: maintain subject recognizability by silhouette and 
    major proportions; no text; no signature; avoid fine line 
    drawing or photoreal textures."""

--- 

### Performing the Style Transfer
[Open image_edits.py](./source/image_edits.py)

In [ ]:
photo_path = Path('resources') / 'sunset.jpg'

In [ ]:
output_path = Path('resources') / 'outputs' / 'styled_sunset.png'

In [ ]:
from source.image_edits import restyle_with_images_api

In [ ]:
restyle_with_images_api(client, photo_path, 
    output_path, '1536x1024', style_transfer_prompt)

In [ ]:
IPython.display.Image(output_path, width=500)

<hr/>


## 3. Style Transfer via Reference Image (Responses API)
* Restyle one image using another as the style reference
* Give Responses API both images and a text instruction
* Responses API uses the image generation tool to perform the transfer
    * Current default model is `gpt-image-2`

### Reference Image

In [ ]:
IPython.display.Image('./resources/style1.jpg', width=200)

### Applying Preceding Image's Style to the Sunset Photo
[Open image_edits.py](./source/image_edits.py)

In [ ]:
reference_image_path = Path('resources') / 'style1.jpg'

In [ ]:
source_image_path = Path('resources') / 'sunset.jpg'

In [ ]:
output_path = (Path('resources') / 'outputs' /
    'styled_sunset_from_style1_image.png')

In [ ]:
from source.image_edits import restyle_with_responses_api

In [ ]:
restyle_with_responses_api(client, reference_image_path, 
    source_image_path, output_path, size='1536x1024')

In [ ]:
IPython.display.Image(output_path, width=500)

---
######

# Additional Info
---

## C2PA Metadata and Content Authenticity
* Generated images contain
    * **C2PA (Coalition for Content Provenance and Authenticity) metadata** 
    * **SynthID watermarks**
* C2PA — open standard developed in an effort to help
    * Verify origins and edit history of digital content 
    * Prevent disinformation
* SynthID adds an image-level watermark
* Not guaranteed
    * **Metadata and watermarks can be stripped and degraded**

### Checking Provenance and Authenticity 
* https://contentcredentials.org/verify
    * Upload a digital media file to inspect its C2PA metadata
    * Verify who or what created digital media
    * How it has changed over time
    * Images may/may not contain C2PA metadata
* https://openai.com/verify
    * Determines if OpenAI model created the image 
* References
    * https://openai.com/verify
    * https://https://contentcredentials.org/
  
---

## OpenAI Images API Notes 
* `background='transparent'` for transparent backgrounds
    * Transparent output requires a format that supports transparency, such as PNG or WebP
    * **Image-generation guide says `gpt-image-2` does not currently support transparent backgrounds**
* Images API edit requests require a `prompt`
    * A `mask` is optional and is used with a prompt for targeted edits/inpainting
* Responses API style-transfer workflows are more flexible when you want the model to reason over multiple input images plus text in one conversation
* Official references:
    * https://developers.openai.com/api/docs/guides/image-generation
    * https://developers.openai.com/api/reference/resources/images/methods/edit
---


## How Do OpenAI Models Ensure Images Are Original?
We were curious how OpenAI ensures images are original, so we asked ChatGPT. It responded:
* Training Process—The model is trained on large collections of image–text pairs. Instead of copying or retrieving existing images, it learns patterns in textures, shapes, colors, and compositions. When prompted, it generates images by sampling from its learned internal representation—this process creates new combinations rather than reproducing stored pictures
* No Database Lookups—gpt-image-1 does not search the Internet or pull images from a library. It doesn't retrieve or remix copyrighted works directly; instead, it generates pixels from scratch based on the statistical relationships it learned during training
* Randomness and Variation—Each generation includes stochastic sampling (randomness). Even with the same prompt, multiple images can differ in details, ensuring that outputs are not deterministic copies of training examples
* Image Generation Safeguards and Recommendations
    * OpenAI implements filters/safety layers to reduce the chance of producing copyrighted, sensitive, or harmful material
    * The model avoids creating near-duplicates of specific, recognizable works or logos when prompted
    * Human Oversight—For critical applications, OpenAI recommends human review and reverse image searches to confirm that the output isn't too close to a copyrighted image
* In short, models produce synthetic, original images guided by your prompt. They don't copy or retrieve existing artworks but instead generate new imagery from learned patterns.

# Follow-Up References
* https://developers.openai.com/api/docs/guides/image-generation
* https://developers.openai.com/api/reference/resources/images/methods/generate
* https://developers.openai.com/api/reference/resources/images/methods/edit


---
&copy; 2026 by Deitel & Associates, Inc. All Rights Reserved. 